# CSTR Optimal Control Tutorial

## Interactive Pyomo Implementation

This notebook provides an **interactive tutorial** for solving an optimal control problem for a Continuous Stirred Tank Reactor (CSTR) using Pyomo.

### Learning Objectives
1. Understand the CSTR dynamics
2. Learn how to formulate optimal control in Pyomo
3. Solve and visualize results
4. Experiment with different parameters

## Step 1: Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyomo.environ import *
from pyomo.opt import SolverFactory

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✓ All libraries imported successfully!")

## Step 2: Define Parameters

These are the physical and operational parameters of our CSTR system.

**Try modifying these values to see how they affect the solution!**

In [ ]:
class Parameters:
    """CSTR Model Parameters"""
    
    # === TIME DISCRETIZATION ===
    dn = 0.05           # ODE time step [hr]
    dt = 0.1            # Control time step [hr]
    tf = 1.0            # Total horizon [hr]
    
    N = int(tf / dn) + 1    # ODE steps
    K = int(tf / dt) + 1    # Control steps
    Kinc = int(dt / dn)     # ODE steps per control
    
    # === PHYSICAL PARAMETERS ===
    dH = -31652.0       # Heat of reaction [kJ/mol]
    U = 1893.0          # Heat transfer coeff [kJ/(hr·m²·K)]
    Ea_R = 8375.0       # Activation energy/R [K]
    k0 = 4.08e10        # Pre-exponential [hr⁻¹]
    
    # === FLUID PROPERTIES ===
    rho = 800.9         # Reactor density [kg/m³]
    Cp = 0.9691         # Reactor heat capacity [kJ/(kg·K)]
    rhoj = 997.95       # Jacket density [kg/m³]
    Cj = 1.292          # Jacket heat capacity [kJ/(kg·K)]
    
    # === OPERATING CONDITIONS ===
    FR = 2.832          # Flow rate [m³/hr]
    TR_in0 = 333.0      # Initial inlet temp [K]
    deltaTR_in = 0.5    # Temp disturbance [K]
    TJ_in = 294.1       # Jacket inlet temp [K]
    CR_in = 10.89       # Inlet concentration [mol/m³]
    
    # === BOUNDS ===
    TR_min, TR_max = 333.0, 360.8
    TJ_min, TJ_max = 300.0, 1110.0
    FJc_min, FJc_max = 20.553, 20.6
    DR_min, DR_max = 3.0, 6.0
    HR_min, HR_max = 4.0, 10.0
    Kc_min, Kc_max = -30.0, 10.0
    tau_min, tau_max = 0.1, 4.0
    
    # === INITIAL GUESSES ===
    CR_opt = 0.275
    TR_opt = 333.0
    TJ_opt = 330.0
    FJ_opt = 20.554
    Kc_opt = -30.0
    tau_I_opt = 0.873
    HR_opt = 10.0
    DR_opt = 5.342
    
    tol = 5e-4          # Endpoint tolerance
    n_tol = 2           # Endpoint steps

p = Parameters()
print(f"✓ Parameters defined!")
print(f"  - Time horizon: {p.tf} hr")
print(f"  - ODE steps: {p.N}")
print(f"  - Control steps: {p.K}")

## Step 3: Build the Optimization Model

This is where we define our decision variables, objective function, and constraints.

In [ ]:
# Import the complete model from the main script
from cstr_optimal_control import build_cstr_model

# Build the model
model, params = build_cstr_model()

print("✓ Model built successfully!")
print(f"\nModel Statistics:")
print(f"  - Number of variables: {len(list(model.component_data_objects(Var)))}")
print(f"  - Number of constraints: {len(list(model.component_data_objects(Constraint)))}")

## Step 4: Solve the Optimization Problem

We'll use the IPOPT solver (Interior Point OPTimizer) to find the optimal solution.

**Note**: This may take 30-60 seconds depending on your computer.

In [ ]:
from cstr_optimal_control import solve_model

# Solve
results = solve_model(model, solver_name='ipopt')

## Step 5: Extract and Display Results

In [ ]:
from cstr_optimal_control import extract_results

# Extract results into DataFrame
df = extract_results(model, params)

# Display first few rows
print("\nTime-series data (first 5 rows):")
df.head()

## Step 6: Visualize Results

Let's create beautiful plots to understand the system behavior!

In [ ]:
from cstr_optimal_control import plot_results

# Create plots
plot_results(df, save_path='cstr_tutorial_results.png')

## Step 7: Analyze Individual Variables

Let's look at each variable more closely.

In [ ]:
# Reactor Temperature Analysis
print("Reactor Temperature (TR) Analysis:")
print(f"  - Initial: {df['TR'].iloc[0]:.4f} K")
print(f"  - Maximum: {df['TR'].max():.4f} K at t={df.loc[df['TR'].idxmax(), 'time']:.2f} hr")
print(f"  - Final: {df['TR'].iloc[-1]:.4f} K")
print(f"  - Change: {df['TR'].iloc[-1] - df['TR'].iloc[0]:.4f} K")

print("\nJacket Temperature (TJ) Analysis:")
print(f"  - Initial: {df['TJ'].iloc[0]:.4f} K")
print(f"  - Maximum: {df['TJ'].max():.4f} K")
print(f"  - Final: {df['TJ'].iloc[-1]:.4f} K")

print("\nControl Flow Rate (FJc) Analysis:")
print(f"  - Initial: {df['FJc'].iloc[0]:.6f} m³/hr")
print(f"  - Maximum: {df['FJc'].max():.6f} m³/hr")
print(f"  - Minimum: {df['FJc'].min():.6f} m³/hr")
print(f"  - Range: {df['FJc'].max() - df['FJc'].min():.6f} m³/hr")

## Step 8: Verify Constraints

Let's check if our endpoint stability constraint is satisfied.

In [ ]:
# Check endpoint tolerance
TR_initial = df['TR'].iloc[0]
TR_final_points = df['TR'].iloc[-p.n_tol:]

print(f"Endpoint Stability Check:")
print(f"  - Initial TR: {TR_initial:.6f} K")
print(f"  - Tolerance: ±{p.tol:.6f} K")
print(f"\n  Last {p.n_tol} points:")
for i, (idx, tr) in enumerate(TR_final_points.items()):
    deviation = tr - TR_initial
    status = "✓" if abs(deviation) <= p.tol else "✗"
    print(f"    {status} TR[{idx}] = {tr:.6f} K (deviation: {deviation:+.6f} K)")

## Step 9: Save Results

In [ ]:
# Save to CSV
df.to_csv('cstr_tutorial_results.csv', index=False)
print("✓ Results saved to 'cstr_tutorial_results.csv'")

# Save optimal parameters
optimal_params = pd.DataFrame({
    'Parameter': ['DR', 'HR', 'Kc', 'tau_I', 'CR[0]', 'TR[0]', 'TJ[0]', 'FJc[0]'],
    'Value': [
        value(model.DR),
        value(model.HR),
        value(model.Kc),
        value(model.tau_I),
        value(model.CR[0]),
        value(model.TR[0]),
        value(model.TJ[0]),
        value(model.FJc[0])
    ],
    'Unit': ['m', 'm', '-', 'hr', 'mol/m³', 'K', 'K', 'm³/hr']
})

optimal_params.to_csv('optimal_parameters.csv', index=False)
print("✓ Optimal parameters saved to 'optimal_parameters.csv'")

print("\nOptimal Parameters:")
optimal_params

## Experiments to Try

Now that you understand the basic model, try these experiments:

### Experiment 1: Different Time Horizons
Go back to Step 2 and change `tf` from 1.0 to 2.0 hours. How does this affect the solution?

### Experiment 2: Coarser Discretization
Change `dn = 0.1` and `dt = 0.2` for a faster (but less accurate) solution.

### Experiment 3: Different Objectives
In the `build_cstr_model()` function, try changing the objective to:
- Minimize equipment cost: `1916 * model.DR**1.66 * model.HR**0.802`
- Minimize flow rate: `model.FJc[0]`

### Experiment 4: Different Bounds
Make the control bounds wider: `FJc_min, FJc_max = 15.0, 25.0`

### Experiment 5: Disturbance Magnitude
Change `deltaTR_in` from 0.5 to 2.0 K and see how the controller responds.

## Summary

Congratulations! You've successfully:
1. ✓ Formulated an optimal control problem in Pyomo
2. ✓ Solved a nonlinear optimization with dynamics
3. ✓ Analyzed and visualized the results
4. ✓ Learned about CSTR control and PI controllers

### Key Takeaways
- **Optimal control** combines optimization with differential equations
- **Backward Euler** is a simple method for discretizing ODEs
- **PI controllers** can be optimized along with design parameters
- **Pyomo** makes it easy to formulate complex optimization problems

### Next Steps
- Try the experiments suggested above
- Read the full documentation in `README_PYOMO.md`
- Extend the model with your own features
- Apply similar techniques to your own problems!